### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [9]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [10]:
@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [11]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [12]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [13]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [14]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)

In [15]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")

In [16]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [17]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some key advantages of using AutoGen for your AI Agent project:

1. **Multi-Agent Framework**: AutoGen allows the definition of multiple AI agents, each with specific roles and capabilities. This enables efficient task delegation and collaboration among agents.

2. **Simplified Development**: It significantly reduces the complexity of developing applications with AI, particularly in managing communication and coordination between agents.

3. **Integration Capabilities**: AutoGen can seamlessly integrate with APIs and external tools, enhancing functionality and allowing your AI agents to interact with a broader ecosystem.

4. **Natural Language Communication**: The use of natural language for inter-agent communication eliminates the need for custom protocols, making coordination simpler and more intuitive.

5. **Customizability**: Agents can be tailored to specific requirements, ensuring that they meet the unique demands of your project.

6. **Feedback Loop Efficiency**: AutoGen enhances the feedback loop among agents, which helps in quick iterations and improvements during the development process.

7. **Support for AI Orchestration**: It provides robust support for orchestrating AI workflows, making automation more efficient and manageable.

8. **Real-World Workflow Automation**: AutoGen is designed to automate complex workflows, driving real business value by increasing productivity and reducing manual effort.

Overall, AutoGen simplifies the development of AI solutions, fosters collaboration, and enhances functionality, making it a compelling choice for your project.

TERMINATE

## Cons of AutoGen:
Here are some cons associated with using AutoGen in an AI Agent project:

1. **Complexity Management**: As project complexity grows, managing workflows with multiple nested chains can become increasingly complicated, leading to potential errors and maintenance challenges.

2. **Learning Curve**: Users often face a steep learning curve with AutoGen, especially due to frequent updates and the need to understand its configurations and workflows.

3. **Inconsistent Outputs**: The framework can produce inconsistent outputs, which complicates debugging and reliability in production settings.

4. **Cost at Scale**: Running AutoGen workflows can become costly, especially if not managed carefully, leading to high operational expenses as project demands increase.

5. **Abstraction Overhead**: The framework might introduce unnecessary abstraction layers that can hinder performance and make processes less transparent.

6. **Documentation Gaps**: Users may encounter insufficient documentation, making it difficult to troubleshoot issues or fully exploit the framework's capabilities.

These factors could significantly impact decision-making when considering AutoGen for your project. 

TERMINATE



## Decision:

Based on the provided research, I recommend using AutoGen for the project. The significant advantages, such as its multi-agent framework, simplified development, and integration capabilities, provide strong justification for its adoption. While there are valid concerns regarding complexity management, learning curves, and potential costs, the benefits of enhanced collaboration, workflow automation, and customization outweigh these drawbacks. Additionally, the ability to drive real business value and increase productivity could lead to long-term gains that mitigate initial challenges.

TERMINATE

In [18]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [19]:
await host.stop()